In [71]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.config import DATA_DIR, FEWSHOT_CACHE_DIR, PROMPT_DIR, GREEDY_CONFIG
import json
from src.extractor import LabelTransformConfig, prepare_label_tokens, _parse_parent_annotations
    
from src.tokenizer_utils import tokenize, decode
from src.htmlLabel import simplified_to_normal_form
from src.models import get_messages
from tqdm import tqdm


### Choose the prompting configuration

In [72]:
filename = "2002SCC33"
split = "dev"
filepath = Path(DATA_DIR) / "annotated" / split / f"{filename}.html"
#filepath = Path("output")  / f"{filename}_0.html"

#filepath = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_2.html"
with open(filepath, "r", encoding="utf-8") as f:
    html_content = f.read()



#### Common Few SHot Selection

In [78]:
fewshot_method = "random"   # "greedy" | "random"

fewshot_filename = f"examples_coref_{fewshot_method}"

with open(FEWSHOT_CACHE_DIR / f"{fewshot_filename}.json", "r", encoding="utf-8") as f:
    fewshot_file_content = json.load(f)

fewshot_examples = [
    (
        json.loads(example["input"]),
        example["output"],
    )
    for example in fewshot_file_content["examples"]
]
print("fewshot examples from :", fewshot_filename)


fewshot examples from : examples_coref_random


##### Few Shot processing step

In [101]:
from src.rpr import ReferenceProfileRegistry
from src.htmlLabel import ReferenceMention
example = fewshot_examples[0]
input, output = example

rpr = ReferenceProfileRegistry.from_dict(input["profileRegistry"])


In [87]:
print(rpr.to_dict(attributes=["main_title", "docid", "alternative_titles", "citations", "fragments_mentioned"]))

{'profiles': [{'main_title': 'Canada (Minister of Citizenship and Immigration) v. Vavilov', 'docid': 'Vavilov SCC 2019', 'alternative_titles': {}, 'citations': {'2019 SCC 65': '0', '[2019] 4 S.C.R. 653': '0', '2019 SCC\xa065': '6'}, 'fragments_mentioned': {'para. 246': '1764', 'para. 277': '1778', 'para. 31': '1857', 'paras. 288': '2047', '289': '2047', '291': '2047', 'para. 13': '2051', 'paras. 89-96': '2291', 'para.\xa0285': '2364', 'paras. 199': '2598', '201': '2598', 'para. 253': '2601', 'para. 284': '2603'}}, {'main_title': 'Bell Canada v. Canada (Attorney General)', 'docid': 'Bell Canada SCC 2019', 'alternative_titles': {}, 'citations': {'2019 SCC 66': '20', '[2019] 4 S.C.R. 845': '20'}, 'fragments_mentioned': {}}, {'main_title': 'Dunsmuir v. New Brunswick', 'docid': 'Dunsmuir', 'alternative_titles': {'Dunsmuir’s': '28', 'Dunsmuir': '59'}, 'citations': {'2008 SCC 9': '24', '[2008] 1 S.C.R. 190': '24'}, 'fragments_mentioned': {'paras. 34-50': '1276', 'paras. 51-64': '1278', 'para.

In [ ]:
def format_profile_for_prompt(profile_dict: dict, max_fragments: int = 10) -> dict:
    """
    Take one profile's to_dict() output (with tracked fields still in
    {value: first_seen_id} form) and turn it into a prompt-friendly dict:
    - tracked fields become plain lists of their keys (ids dropped)
    - empty lists are omitted entirely
    - fragments_mentioned is truncated to the last `max_fragments` items
    """
    tracked_fields = ("alternative_titles", "citations", "fragments_mentioned", "authors")

    formatted = {}
    for key, value in profile_dict.items():
        if key in tracked_fields:
            values_list = list(value.keys()) if isinstance(value, dict) else list(value)
            if key == "fragments_mentioned":
                values_list = values_list[-max_fragments:]
            if not values_list:
                continue  # drop empty lists
            formatted[key] = values_list
        else:
            formatted[key] = value

    return formatted


def example_to_string(example_input: dict, docid, doctype, max_fragments: int = 10) -> str:
    """
    Given one fewshot example's `input` dict (with keys "input_mention",
    "context", "profileRegistry"), build a single formatted string
    describing the input. Output mention is intentionally not included.
    """
    rpr = ReferenceProfileRegistry.from_dict(example_input["profileRegistry"])

    attributes = ["doc_type", "main_title", "alternative_titles",
                  "citations", "fragments_mentioned", "authors"]

    """
    filtered_rpl = sample_reference_profile_subset(
            rpr,                 # your ReferenceProfileRegistry
            docid,                # the docid to check for
            doc_type=doctype,            # optional doc_type filter
            include_docid="yes",  # "yes" | "no" | "random"
            length=None,             # OR min_length=2, max_length=8
            min_length=0,
            max_length=5,
            seed=42,
            p=1,                # only used when include_docid="random"
            must_include_main_title=True,        # if main_title is missing, the profile is excluded
        )
        """
    profiles_formatted = [
        format_profile_for_prompt(profile.to_dict(attributes=attributes), max_fragments=max_fragments)
        for profile in rpr
    ]

    lines = []
    lines.append(f"Input mention: {example_input['input_mention']}")
    lines.append(f"Context: {example_input['context']}")
    lines.append("Reference Profile Registry:")
    for i, profile in enumerate(profiles_formatted):
        lines.append(f"  Profile {i}: {profile}")

    return "\n".join(lines)

In [110]:
output

'"<decision docid=\\"Canada v. Craig\\"><title>Craig</title>, at <fragment>para. 24</fragment></decision>"'

In [121]:
input_["input_mention"].split(">")[0][1:]

'legislation'

In [138]:
for i in range(6):
    example = fewshot_examples[i]
    input_, output = example
    print(f"Example {i}:")
    print(example_to_string(input_, docid=output.split('docid=\\"')[1].split('\\"')[0], doctype=input_["input_mention"].split(">")[0][1:]))
    print("-" * 50)

Example 0:


ValueError: docid 'Canada v. Craig' not found in the given ReferenceProfileRegistry

In [22]:
import re

from src.htmlLabel import from_simplified

final_fewshot = []
parents_dict = _parse_parent_annotations(total_output_text)
for parent_name, annotations in parents_dict.items():
    for annotation in annotations:
        print(annotation)
        label = from_simplified(simplified_token=tokenize(annotation)[0], label_type='manual_label')
        name = label.name
        docid = label.attributes["docid"]

        filtered_rpl = sample_reference_profile_subset(
            rpl,                 # your ReferenceProfileList
            docid,                # the docid to check for
            doc_type=name,            # optional doc_type filter
            include_docid="random",  # "yes" | "no" | "random"
            length=None,             # OR min_length=2, max_length=8
            min_length=0,
            max_length=5,
            seed=42,
            p=0.8,                # only used when include_docid="random"
        )
        input_annotation = decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))
        input_rpl = f"Profiles registry : {', '.join(str(profile) for profile in filtered_rpl.profiles)}"
        final_fewshot.append((f"{input_annotation}\n\n{input_rpl}", annotation))

<decision docid="Volvo Canada"><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>
<decision docid="Toronto"><title>Toronto (City)</title>, at <fragment>paras. 94-95</fragment></decision>
<decision docid="Council of Canadians with Disabilities"><title>VIA Rail</title>, at <fragment>para. 101</fragment></decision>
<decision docid="Mason"><title>Mason v. Minister of Citizenship and Immigration</title>, <citation>2019 FC 1251</citation>, at <fragment>para. 22</fragment></decision>
<decision docid="Irwin Toy"><title>Irwin Toy Ltd. v. Quebec
(Attorney General)</title>, <citation>[1989] 1 S.C.R. 927</citation>, <citation>39 C.R.R. 193</citation></decision>
<decision docid="Généreux"><title>R. v. Généreux</title>, <citation>[1992] 1
S.C.R. 259</citation> at <fragment>310</fragment>, <citation>8 C.R.R. (2d) 89</citation> at <fragment>p. 124</fragment></decision>
<decision docid="Kimble"><title>Kimble</title>

In [16]:
final_fewshot

[("<decision><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>\n\nProfiles registry : {'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Volvo Canada Ltd. v. U.A.W., Local 720', 'docid': 'Volvo Canada', 'alternative_titles': [], 'citations': ['[1980] 1 S.C.R. 178'], 'fragments_mentioned': ['p. 214'], 'authors': []}",
  '<decision docid="Volvo Canada"><title>Volvo Canada Ltd. v. U.A.W., Local 720</title>, <citation>[1980] 1 S.C.R. 178</citation>, at <fragment>p. 214</fragment></decision>'),
 ("<decision><title>Toronto (City)</title>, at <fragment>paras. 94-95</fragment></decision>\n\nProfiles registry : {'doc_type': 'decision', 'jurisdiction': None, 'main_title': 'Toronto (City) v. C.U.P.E., Local 79', 'docid': 'Toronto', 'alternative_titles': ['Toronto (City)'], 'citations': ['2003 SCC 63', '[2003] 3 S.C.R. 77'], 'fragments_mentioned': ['para. 62', 'para.\\xa070', 'para. 15', 'para. 131'

In [131]:
import random

from src.rpr import ReferenceProfileRegistry


def sample_reference_profile_subset(
    rpl: ReferenceProfileRegistry,
    docid,
    doc_type: str = None,
    include_docid="yes",
    length: int = None,
    min_length: int = None,
    max_length: int = None,
    seed: int = None,
    p: float = 0.7,
    must_include_main_title: bool = True
) -> ReferenceProfileRegistry:
    """
    Build a random subset of `rpl` as a new ReferenceProfileRegistry.

    Parameters
    ----------
    rpl : ReferenceProfileRegistry
        The full list of profiles to sample from.
    docid :
        The docid we care about when deciding inclusion.
    doc_type : str, optional
        If given, only profiles with this `doc_type` are considered for the subset.
    include_docid : {"yes", "no", "random"}
        - "yes":    the profile with `docid` is forced into the subset.
        - "no":     the profile with `docid` is forced OUT of the subset.
        - "random": the profile with `docid` is included with probability `p`.
    length : int, optional
        Exact size of the returned subset. If given, takes priority over
        min_length/max_length.
    min_length, max_length : int, optional
        If `length` is not given, the subset size is drawn uniformly from
        [min_length, max_length] (inclusive), using `seed`.
    seed : int, optional
        Seed for all the random choices made in this function (size choice,
        whether to include the target docid in "random" mode, and which
        other profiles fill the rest of the subset). Uses a local
        random.Random instance, so global random state is untouched.
    p : float, default 0.7
        Probability of including the target docid's profile when
        include_docid="random". Ignored otherwise.

    Returns
    -------
    ReferenceProfileRegistry
        A new list containing the sampled subset of profiles.

    Raises
    ------
    ValueError
        If include_docid is not one of "yes"/"no"/"random"; if include_docid
        is "yes" but no profile with `docid` exists in `rpl`; if neither
        `length` nor a valid (min_length, max_length) pair is given; or if
        the requested subset size is larger than what's available.
    """
    if include_docid not in ("yes", "no", "random"):
        raise ValueError(
            f"include_docid must be 'yes', 'no', or 'random', got {include_docid!r}"
        )

    rng = random.Random(seed)

    all_profiles = list(rpl)
    if doc_type is not None:
        all_profiles = [prof for prof in all_profiles if prof.doc_type == doc_type]
    if must_include_main_title:
        all_profiles = [prof for prof in all_profiles if prof.main_title is not None]
    target_profile = rpl.get_profile_by_docid(docid)

    if include_docid == "yes" and target_profile is None:
        raise ValueError(f"docid {docid!r} not found in the given ReferenceProfileRegistry")

    # Decide, for this call, whether the target profile should be forced in,
    # forced out, or absent because it doesn't exist.
    force_include_target = False
    force_exclude_target = False

    if target_profile is None:
        # Nothing to force either way; "no" and "random" are trivially satisfied.
        force_exclude_target = True
    elif include_docid == "yes":
        force_include_target = True
    elif include_docid == "no":
        force_exclude_target = True
    else:  # "random"
        if rng.random() < p:
            force_include_target = True
        else:
            force_exclude_target = True

    # Pool of profiles eligible to fill the "free" slots of the subset
    # (everything except the target profile, which is handled separately).
    other_profiles = [prof for prof in all_profiles if prof is not target_profile]

    # Work out the desired subset size.
    # Max possible size of the final subset given the forced inclusion/exclusion:
    if force_include_target:
        max_possible = 1 + len(other_profiles)
    else:
        max_possible = len(other_profiles)

    if length is not None:
        subset_size = length
    else:
        if min_length is None or max_length is None:
            raise ValueError(
                "Either `length`, or both `min_length` and `max_length`, must be provided"
            )
        if min_length > max_length:
            raise ValueError("min_length cannot be greater than max_length")
        subset_size = rng.randint(min_length, max_length)

    if subset_size < 0:
        raise ValueError("Computed subset size is negative")

    # How many additional (non-target) profiles do we need to fill the subset?
    remaining_slots = subset_size - 1 if force_include_target else subset_size
    remaining_slots = max(remaining_slots, 0)

    chosen_others = rng.sample(other_profiles, min(remaining_slots, len(other_profiles))) if remaining_slots > 0 else []

    subset_profiles = list(chosen_others)
    if force_include_target:
        subset_profiles.append(target_profile)

    # Shuffle so the target profile (if forced in) isn't always last.
    rng.shuffle(subset_profiles)

    result = ReferenceProfileRegistry()
    for prof in subset_profiles:
        result.add_profile(prof)

    return result

In [4]:
nb_fewshot_examples = 6
spans_in_context = False

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_attributes=["labelname"]
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=True,
    keep_attributes=["labelname", "docid"]
)


# Transform the output in it simplified form
final_fewshot = []
total_output_text = ""
for example in fewshot_examples:
    input, output, list_reference_profile = example

    input_tokens = tokenize(input)
    transformed_input_tokens = prepare_label_tokens(input_tokens, input_label_config)

    output_tokens = tokenize(output)
    transformed_output_tokens = prepare_label_tokens(output_tokens, output_label_config)

    if spans_in_context:
        final_fewshot.append((decode(transformed_input_tokens), decode(transformed_output_tokens)))

    if not spans_in_context:

        total_output_text += "|||" + decode(transformed_output_tokens)

final_fewshot = final_fewshot[:nb_fewshot_examples]


if not spans_in_context:
    parents_dict = _parse_parent_annotations(total_output_text)
    for parent_name, annotations in parents_dict.items():
        for annotation in annotations:
            input = f"{decode(prepare_label_tokens(simplified_to_normal_form(tokenize(annotation),label_type="manual_label"), input_label_config))} Profiles registry : {', '.join(list_reference_profile["profiles"])}"
            if input != annotation:
                final_fewshot.append((input, annotation))

TypeError: sequence item 0: expected str instance, dict found

In [35]:
final_fewshot

[('<legislation><citation>R.S.B.C. 1996, c. 418</citation>, <fragment>s. 159</fragment></legislation>',
  '<legislation docid="Securities Act"><citation>R.S.B.C. 1996, c. 418</citation>, <fragment>s. 159</fragment></legislation>'),
 ('<legislation><fragment>Section 7</fragment> of the <title>Charter</title></legislation>',
  '<legislation docid="Charter"><fragment>Section 7</fragment> of the <title>Charter</title></legislation>'),
 ('<legislation><fragment>s. 7</fragment> of the <title>Charter</title></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment> of the <title>Charter</title></legislation>'),
 ('<legislation><fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment></legislation>'),
 ('<legislation><fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"><fragment>s. 7</fragment></legislation>'),
 ('<legislation> <fragment>s. 7</fragment></legislation>',
  '<legislation docid="Charter"> <fragment>s. 7

#### Common Prompt loading

In [29]:
prompt_filename = "coref_long.txt"

with open(PROMPT_DIR / prompt_filename, "r", encoding="utf-8") as f:
    system_prompt = f.read()

print("system_prompt used : ", prompt_filename)

system_prompt used :  coref_long.txt


#### Assistant loading

In [7]:
MODEL_MAPPING_NAME = {
    "qwen7b": "Qwen2.5-7B-Instruct",
    "qwen32b": "Qwen2.5-32B-Instruct",
    "gpt-5.2": "gpt-5.2",
    "phi-4": "phi-4",
    "saul-54b": "SaulLM-54B-Instruct"
}

from src.models import AssistantFactory

model = "gpt-5.2"

if model == "gpt-5.2":
        assistant = AssistantFactory.create_from_config({
            "type": "openai",
            "model_name": model,
            "temperature": 1,
        })
else:
    assistant = AssistantFactory.create(MODEL_MAPPING_NAME[model])


#### Chunk output controle

In [8]:
def process_output(generated, token_chunk, allowed_labels, assistant, with_fallback: bool = True):

    from src.output_control.processor import OutputProcessor
    from src.output_control.fallback import FallbackHandler 
    from src.output_control.verification import VerificationResult 

    controller = OutputProcessor()
    fallback_handler = FallbackHandler(processor=controller)
    
    corrected_generated_tokens, status = controller.process(
        raw_llm_output=generated,
        original_chunk=token_chunk,
        allowed_labels=allowed_labels
    )

    if status.passed:
        return corrected_generated_tokens, status

    if not with_fallback:
        return token_chunk, status
    
    
    corrected_generated_tokens, status_dict = fallback_handler.handle_failure(
        assistant=assistant,
        corrected_output=corrected_generated_tokens,
        original_chunk=token_chunk,
        initial_status=status,
        allowed_labels=allowed_labels,
        fallback_prompt_filename="fallback.txt"
    )
    # Convert dict to VerificationResult
    status = VerificationResult(
        passed=status_dict.get('passed', False),
        error_type=status_dict.get('error_type'),
        details=status_dict.get('error_details'),
        tokens=corrected_generated_tokens
    )
    
    return corrected_generated_tokens, status

### For DEC1-3 

No chunking needed here, we just need the list of mention already labeled

In [32]:
allowed_labels = ["decision", "legislation", "secondary sources", "title", "citation", "source", "authors", "fragment"]

#### Convert into tokens

In [30]:
from src import extract_body, tokenize, clean_tokens
tokens = tokenize(html_content)


#### Get already extracted mention

In [31]:

from src.extractor import build_processing_segments
from src.extractor import get_list_of_mention
from src.models import get_messages
from tqdm import tqdm
from tqdm import tqdm

parent_mentions = get_list_of_mention(
        tokens=tokens,
        keep_labels=["decision", "legislation", "secondary sources"],
        label_type="manual_label"  # Process manual labels from parent extraction
    )

print(f"Found {len(parent_mentions)} parent mentions to process")

segments = build_processing_segments(tokens, parent_mentions)

print(f"Built {len(segments)} token segments "
        f"({sum(s['process'] for s in segments)} to process)")

Found 328 parent mentions to process
Built 657 token segments (328 to process)


#### Main processing function

In [ ]:
config = LabelTransformConfig(
    use_simplified=False,
    switch_type=False,
    keep_attributes=["labelname"]
) # We only remove the attribute
 

failed_count = 0

for idx, segment in enumerate(tqdm(segments, desc="Processing mentions")):
        if not segment["process"]:
            continue


        mention = segment["tokens"]
        html_label = segment["meta"]["label"]

        
        prepared_tokens = prepare_label_tokens(mention, input_label_config)
        user_input = decode(prepared_tokens)

        filtered_fewshot = []
        for example in final_fewshot:
            if example[0].startswith(f"<{html_label.name}>"):
                filtered_fewshot.append(example)
        messages = get_messages(system_prompt=system_prompt, user_input=user_input, fewshot_examples=filtered_fewshot, has_system_role=assistant.has_system_role)


        generated = assistant.generate(messages=messages)
        #print(generated)
        
        corrected_generated_tokens, status = process_output(generated=generated, token_chunk=prepare_label_tokens(mention, config), allowed_labels=allowed_labels, assistant=assistant, with_fallback=False)
        #print(decode(corrected_generated_tokens))
        if not status.passed:
            failed_count += 1

        segment["tokens"] = corrected_generated_tokens

processed_tokens = [
        token
        for segment in segments
        for token in segment["tokens"]
    ]

Processing mentions:   0%|          | 0/557 [00:00<?, ?it/s]

Processing mentions: 100%|██████████| 557/557 [07:46<00:00,  1.19it/s]


#### Post Processing : tokens to HTML

In [43]:
from src.post_processing.main import tokens_to_html_after_decomposed1_3_prompting
processed_html_content = tokens_to_html_after_decomposed1_3_prompting(processed_tokens, html_content)

   ✓ HTMLs match after normalization (ignoring auto_label tags and formatting artifacts)


#### Save File

In [45]:
output_filename = PROJECT_ROOT / Path("output") / "dev_gpt5.2_DEC_fs6_greedy_sentence_long" / filename / f"{filename}_final.html"
#output_filename = Path("output") / f"{filename}_1.html"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(processed_html_content)

# TEST PRECOMPUTE FEWSHOT

In [6]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [14]:
"""
Select and cache few-shot examples.
Requires chunk cache and pattern dict to already exist.

Usage:
    python precompute_fewshot_examples.py
    python precompute_fewshot_examples.py --method random --n 20
    python precompute_fewshot_examples.py --force
    python precompute_fewshot_examples.py --compare  # Generate both greedy & random + comparison plot
"""
import argparse
import json
from configs.config import PROFILE_CACHE_DIR, DATA_DIR, FEWSHOT_N

from src.transforme_utils import prepare_label_tokens, clean_tokens
from src.html_utils import extract_body
from src.tokenizer_utils import tokenize, decode
from src.ann_extractor import extract_parent_level_annotations

from src.transforme_utils import LabelTransformConfig

from pathlib import Path


def get_html_files(split: str) -> dict[str, str]:
    folder = DATA_DIR / "annotated" / split
    if not folder.is_dir():
        print(f"⚠  Not found, skipping: {folder}")
        return {}
    files = {}
    for path in folder.iterdir():
        if path.suffix.lower() not in {".html", ".htm"}:
            continue
        files[path.stem] = path.read_text(encoding="utf-8")
    return files


def _load_extraction_examples(fs_min_tokens: int) -> list[dict]:
    """Chunk train HTML files and extract few-shot (input, output) pairs."""
    from src.chunkers.factory import ChunkerFactory
    from src.extractor import extract_few_shot_examples

    # Create input label config for processing
    input_label_config = LabelTransformConfig(
        use_simplified=False,
        switch_type=False,
        remove_attributes=["verified", "style"],
    )

    examples = []
    for filename, html in get_html_files("train").items():
        chunks = ChunkerFactory.get_chunks(
            html, method="paragraph",
            filename=filename, min_tokens=fs_min_tokens
        )
        examples.extend(extract_few_shot_examples(chunks, input_label_config, source_file=filename))
    
    return examples

import json

In [29]:
def get_mention_upper_context(html: str, mention, max_tokens: int = 500) -> str:
    """
    Find `mention` inside `html`, and return up to `max_tokens` tokens
    of context immediately preceding it, decoded back to a string.

    Args:
        html: the full document html.
        mention: a mention object with `.html_tag` (has `.attributes["id"]`)
                  and `.html_str` (the raw html snippet for this mention).
        max_tokens: maximum number of preceding tokens to include.

    Returns:
        Decoded string of the context window preceding the mention.
    """
    tokens = tokenize(html)

    start_idx = _find_mention_token_start(tokens, mention)

    if start_idx is None:
        raise ValueError(f"Could not locate mention with id={mention.html_tag.attributes.get('id')} in tokenized html")

    context_start = max(0, start_idx - max_tokens)
    context_tokens = tokens[context_start:start_idx]

    return decode(context_tokens)


def _find_mention_token_start(tokens, mention):
    """
    Locate the index of the first token belonging to `mention` inside `tokens`.
    """
    tag = mention.html_tag

    for i, tok in enumerate(tokens):


        if str(tag) == tok:
            return i

    return None

In [45]:
from src.extractor import tokenize
from src.rpr import ReferenceProfileRegistry
from tqdm import tqdm

input_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname"],
)

output_label_config = LabelTransformConfig(
    use_simplified=True,
    switch_type=False,
    keep_attributes=["labelname", "docid"],
)

examples = []
for filename, html in get_html_files("train").items():

    cleaned_html = decode(clean_tokens(tokenize(html), keep_manual_label=True, keep_auto_label=True))

    
    cache_path = PROFILE_CACHE_DIR / "train" / f"{Path(filename).stem}.json"
    if not cache_path.exists():
        continue
    with open(cache_path, "r", encoding="utf-8") as f:
        registry_dict = json.load(f)

    rpr = ReferenceProfileRegistry.from_dict(registry_dict)

    mentions = extract_parent_level_annotations(cleaned_html)

    for mention in tqdm(mentions):
        mention_id_str = mention.html_tag.attributes.get("id")
        if mention_id_str is None:
            continue
        mention_id = int(mention_id_str)

        context = get_mention_upper_context(cleaned_html, mention, max_tokens=FEWSHOT_N)
        #print(mention)

        input_mention = decode(prepare_label_tokens(tokenize(mention.html_str), input_label_config))

        docid = mention.html_tag.attributes["docid"]
        
        profile = rpr.get_profile_by_docid(docid)
        if profile is None:
            print(f"Warning: docid {docid} not found in registry for mention id {mention_id}. Skipping this mention.")
            continue

        mention.html_tag.set_attribute("docid", profile.main_title)  # replace docid by the main_title
  
        #print(mention.html_tag)
        #print(mention.html_str)
        output_mention = decode(prepare_label_tokens(tokenize(mention.html_str), output_label_config))

        snapshot_filtered = rpr._filter_registry_before(mention_id)
        snapshot_filtered = snapshot_filtered._filter_by_doctype(mention.html_tag.name)  # Filter the snapshot by the mention's doc_type

        examples.append({
            "input": json.dumps({"input_mention": input_mention, "profileRegistry": snapshot_filtered.to_dict(), "context": context}, ensure_ascii=False),
            "output": json.dumps(output_mention, ensure_ascii=False),
        })

 15%|█▍        | 25/167 [00:00<00:03, 39.46it/s]

 23%|██▎       | 38/167 [00:01<00:03, 37.18it/s]

 56%|█████▌    | 76/136 [00:03<00:03, 18.76it/s]

 22%|██▏       | 321/1446 [01:40<05:33,  3.38it/s]

 31%|███       | 449/1446 [02:14<04:11,  3.97it/s]

 77%|███████▋  | 1120/1446 [05:23<01:33,  3.48it/s]

100%|██████████| 1446/1446 [07:24<00:00,  3.26it/s]
